In [45]:
import networkx as nx
import time
from itertools import combinations
from pathlib import Path

%run 01_algorithms.ipynb

In [46]:
def test_algorithm(algorithm, files, is_exact=False):
    for filepath in files:
        G, terminals, name = parse_stp(filepath)
        print(f'--- {name} ---')
        print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}, Terminals: {len(terminals)}')

        start = time.time()
        if is_exact:
            tree, weight, subsets = algorithm(G, terminals)
            print(f'Subsets checked: {subsets}')
        else:
            tree, weight = algorithm(G, terminals)
        elapsed = time.time() - start

        steiner_points = [v for v in tree.nodes() if v not in terminals]
        print(f'Weight: {weight}')
        print(f'Steiner points used: {steiner_points}')
        print(f'Edges: {list(tree.edges(data=True))}')
        print(f'Time: {elapsed:.4f}s')
        print()


def compare_algorithms(algorithms, files):
    results = []
    for filepath in files:
        G, terminals, name = parse_stp(filepath)
        row = {'instance': name, 'nodes': G.number_of_nodes(), 'terminals': len(terminals)}
        for alg_name, alg_func, is_exact in algorithms:
            start = time.time()
            if is_exact:
                tree, weight, _ = alg_func(G, terminals)
            else:
                tree, weight = alg_func(G, terminals)
            elapsed = time.time() - start
            row[alg_name] = weight
            row[alg_name + '_time'] = elapsed
        results.append(row)

    alg_names = [name for name, _, _ in algorithms]
    header = f"{'Instance':<10} {'N':>4} {'T':>4} | " + "  ".join(f"{name[:6]:>6}" for name in alg_names)
    print(header)
    print('-' * len(header))
    bf_name = alg_names[0]
    for r in results:
        bf = r[bf_name]
        print(f"{r['instance']:<10} {r['nodes']:>4} {r['terminals']:>4} | {bf:>6.0f}", end='')
        for alg_name in alg_names[1:]:
            w = r[alg_name]
            diff = (w - bf) / bf * 100
            if w == bf:
                print(f"  {'*':>6}", end='')
            else:
                print(f"  {w:>3.0f}+{diff:.0f}%", end='')
        print()

    print()
    print("* = optimalno rešenje")